In [22]:
import slideflow as sf
import json
import os
import pandas as pd

# Use absolute path to slides directory
slides_dir = "/workspace/dp-code/.scratch/datasets/breast_cancer"

# Create project
P = sf.Project("proj", create=True)
P.add_source(name="source", slides=slides_dir)

# Create annotations.csv if it doesn't exist or is empty
annotations_path = P.annotations
df = pd.read_csv(annotations_path) if os.path.exists(annotations_path) else pd.DataFrame()

if len(df) == 0:
    print("Creating annotations for MRXS files...")
    mrxs_files = [f for f in os.listdir(slides_dir) if f.endswith('.mrxs')]
    annotations_data = []
    for mrxs_file in mrxs_files:
        slide_name = mrxs_file.replace('.mrxs', '')
        annotations_data.append({
            'slide': slide_name,
            'patient': slide_name,
            'recurrence': 0
        })
    
    df = pd.DataFrame(annotations_data)
    df.to_csv(annotations_path, index=False)
    print(f"Created annotations for {len(mrxs_files)} slides")

# Create dataset
dataset = P.dataset(tile_px=256, tile_um=302)
print(f"Dataset has {len(dataset.slides())} slides")

# Extract tiles with ROI-based filtering
# The ROIs will be automatically loaded from geojson files if they exist
print("\nExtracting tiles with ROI filtering...")
P.extract_tiles(
    tile_px=256,
    tile_um=302,
    roi_method="inside"  # Extract tiles only at ROI centers
)
print("✓ Tile extraction complete!")

[09:47:37] INFO     Saved dataset source source to proj/datasets.json

/opt/venv/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Dataset has 1 slides

Extracting tiles with ROI filtering...


           INFO     Slide reading backend: libvips

           INFO     Filtering tiles by grayspace fraction

...

           INFO     Using 12 processes (pool=spawn)

/opt/venv/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

           INFO     Generating PDF (this may take some time)...

Skipping CSV update; no extraction reports found.


✓ Tile extraction complete!


In [17]:
# Install system dependencies if needed
import subprocess
import sys

print("Installing system dependencies for slide processing...")
try:
    subprocess.run(['apt-get', 'update'], check=True, capture_output=True)
    subprocess.run(['apt-get', 'install', '-y', 'libvips42'], check=True, capture_output=True)
    print("✓ libvips installed successfully")
except Exception as e:
    print(f"Warning: Could not install libvips via apt: {e}")
    print("Trying alternative: openslide")
    try:
        subprocess.run(['apt-get', 'install', '-y', 'libopenslide0', 'libopenslide-dev'], check=True, capture_output=True)
        print("✓ openslide installed")
    except Exception as e2:
        print(f"Warning: Could not install openslide: {e2}")

Installing system dependencies for slide processing...
✓ libvips installed successfully


In [11]:
# Detailed folder inspection
import os
import json

slides_path = ".scratch/datasets/breast_cancer"
print(f"=== Inspecting: {slides_path} ===\n")

if os.path.exists(slides_path):
    # List all files and directories
    items = os.listdir(slides_path)
    print(f"Total items in folder: {len(items)}\n")
    
    for item in sorted(items):
        full_path = os.path.join(slides_path, item)
        if os.path.isfile(full_path):
            size = os.path.getsize(full_path)
            print(f"📄 {item} ({size:,.0f} bytes)")
            
            # If it's geojson, show structure
            if item.endswith('.geojson'):
                with open(full_path) as f:
                    data = json.load(f)
                    print(f"   → Contains {len(data.get('features', []))} features")
        else:
            print(f"📁 {item}/")
            # List subdirectory contents
            sub_items = os.listdir(full_path)
            for sub in sub_items:
                print(f"   ├─ {sub}")

# Now check what Slideflow sees
print("\n=== Slideflow Project Status ===")
print(f"Annotations file: {P.annotations}")
print(f"  Exists: {os.path.exists(P.annotations)}")
print(f"Dataset config: {P.dataset_config}")
print(f"  Exists: {os.path.exists(P.dataset_config)}")

# List all .csv files in parent directories
print("\n=== Searching for CSV files ===")
for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith('.csv'):
            print(f"Found: {os.path.join(root, file)}")

=== Inspecting: .scratch/datasets/breast_cancer ===


=== Slideflow Project Status ===
Annotations file: proj/annotations.csv
  Exists: True
Dataset config: proj/datasets.json
  Exists: True

=== Searching for CSV files ===
Found: ./proj/annotations.csv


In [13]:
# Check current working directory and find slides
import pandas as pd
import os

print(f"=== Current Working Directory ===")
print(f"CWD: {os.getcwd()}\n")

# List available paths
print("=== Checking Possible Paths ===")
possible_paths = [
    ".scratch/datasets/breast_cancer",
    "/workspace/dp-code/.scratch/datasets/breast_cancer",
    "../../.scratch/datasets/breast_cancer"
]

slides_path = None
for path in possible_paths:
    exists = os.path.exists(path)
    print(f"{path}: {exists}")
    if exists:
        slides_path = path
        break

if slides_path is None:
    # Try to find it
    print("\n=== Searching for breast_cancer folder ===")
    for root, dirs, files in os.walk("/workspace"):
        if "breast_cancer" in dirs:
            slides_path = os.path.join(root, "breast_cancer")
            print(f"Found: {slides_path}")
            break

if slides_path and os.path.exists(slides_path):
    print(f"\n=== Using path: {slides_path} ===")
    
    # Check annotations
    annotations_path = P.annotations
    print(f"Annotations file: {annotations_path}")
    print(f"Exists: {os.path.exists(annotations_path)}\n")
    
    if os.path.exists(annotations_path):
        df = pd.read_csv(annotations_path)
        print(f"Current annotations shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        print(f"\nContent:")
        print(df)
    
    # Check what slides are in the directory
    print(f"\n=== Slides in {slides_path} ===")
    slide_files = [f for f in os.listdir(slides_path) if f.endswith('.mrxs')]
    print(f"Found {len(slide_files)} MRXS files:")
    for sf in slide_files:
        print(f"  - {sf}")
    
    # Create/update annotations if needed
    if len(slide_files) > 0 and len(df) == 0:
        print("\n=== Creating Annotations Entry ===")
        # Create annotations for each slide
        new_data = []
        for slide_file in slide_files:
            slide_name = slide_file.replace('.mrxs', '')
            new_data.append({
                'slide': slide_name,
                'patient': slide_name,
                'recurrence': 0
            })
        
        new_df = pd.DataFrame(new_data)
        new_df.to_csv(annotations_path, index=False)
        print(f"Created annotations for {len(new_data)} slides:")
        print(new_df)
else:
    print("\n❌ Could not find breast_cancer folder!")

=== Current Working Directory ===
CWD: /workspace/dp-code/tools

=== Checking Possible Paths ===
.scratch/datasets/breast_cancer: False
/workspace/dp-code/.scratch/datasets/breast_cancer: True

=== Using path: /workspace/dp-code/.scratch/datasets/breast_cancer ===
Annotations file: proj/annotations.csv
Exists: True

Current annotations shape: (0, 4)
Columns: ['patient', 'dataset', 'category', 'slide']

Content:
Empty DataFrame
Columns: [patient, dataset, category, slide]
Index: []

=== Slides in /workspace/dp-code/.scratch/datasets/breast_cancer ===
Found 1 MRXS files:
  - 25.mrxs

=== Creating Annotations Entry ===
Created annotations for 1 slides:
  slide patient  recurrence
0    25      25           0
